# Multi-Modal Scene Audio Generation Pipeline Demo

Step-by-step walkthrough of the full pipeline:
1. Load 4 corner images
2. Stage-1 analysis → structured room report
3. Stage-2 generation → 4–6 new views (sequential conditioning)
4. 3-D reconstruction (COLMAP + MiDaS depth)
5. Scene segmentation (Qwen2.5-VL)
6. Audio generation (MMAudio)
7. Spatial audio positioning & visualisation

## Setup

In [ ]:
import sys
sys.path.append('..')

import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from dotenv import load_dotenv

load_dotenv('../.env')
print('Setup complete')

## 1. Load Input Images

In [ ]:
from src.utils import load_images_from_directory

images, paths = load_images_from_directory('../data/input')
print(f'Loaded {len(images)} images')

fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
if len(images) == 1:
    axes = [axes]
for ax, img, path in zip(axes, images, paths):
    ax.imshow(img)
    ax.set_title(os.path.basename(path))
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Generate Additional Viewpoints

**Stage 1** – `analyze_scene()` sends all 4 corner images to `gemini-2.0-flash` (text only) and returns a structured room report covering: room envelope, fixed architecture, furniture positions, lighting, and photographic style.

**Stage 2** – `generate_additional_views()` injects that report verbatim into every generation prompt alongside all 4 original images. With `chain_views=True` each new view is also fed back as a visual anchor for the next call, reducing cross-view drift.

In [ ]:
from src.image_generation import ImageGenerator

generator = ImageGenerator()

# --- Stage 1: structured analysis ---
print('Stage 1: analysing scene...')
scene_description = generator.analyze_scene(images)
print('\n' + '='*60)
print('SCENE ANALYSIS')
print('='*60)
print(scene_description)

# --- Stage 2: novel view synthesis ---
print('\nStage 2: generating views...')
generated_images = generator.generate_additional_views(
    input_images=images,
    scene_description=scene_description,
    num_views=4,
    output_dir='../data/generated',
    chain_views=True,
)
print(f'Generated {len(generated_images)} views')

n = len(generated_images)
cols = min(n, 2)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 6 * rows))
axes = np.array(axes).flatten()
for ax, img in zip(axes, generated_images):
    ax.imshow(img)
    ax.axis('off')
for ax in axes[n:]:
    ax.set_visible(False)
plt.suptitle('Generated Viewpoints (sequential conditioning)')
plt.tight_layout()
plt.show()

## 3. 3-D Reconstruction

In [ ]:
from src.reconstruction import reconstruct_scene

all_images = images + generated_images

reconstruction_results = reconstruct_scene(
    images=all_images,
    image_paths=paths,
    config={
        'use_colmap': True,
        'use_depth_model': True,
        'depth_model': 'midas_v3_dpt_large',
        'colmap': {
            'feature_type': 'SIFT',
            'matching_method': 'exhaustive',
            'dense_reconstruction': False,
        },
    },
    output_dir='../data/reconstruction',
)

print('Reconstruction results:')
if reconstruction_results.get('depth_maps'):
    print(f'  depth maps: {len(reconstruction_results["depth_maps"])}')
if reconstruction_results.get('sparse_reconstruction'):
    sp = reconstruction_results['sparse_reconstruction']
    print(f'  sparse points: {len(sp.points3D)}')
    print(f'  cameras: {len(sp.images)}')

if reconstruction_results.get('depth_maps'):
    dmaps = reconstruction_results['depth_maps']
    show = min(4, len(dmaps))
    fig, axes = plt.subplots(1, show, figsize=(5 * show, 5))
    if show == 1:
        axes = [axes]
    for ax, dm in zip(axes, dmaps[:show]):
        ax.imshow(dm, cmap='viridis')
        ax.axis('off')
    plt.suptitle('Depth Maps')
    plt.tight_layout()
    plt.show()

## 4. Scene Segmentation (Qwen2.5-VL)

In [ ]:
import torch
from src.segmentation import SceneSegmenter, project_objects_to_3d

segmenter = SceneSegmenter(
    quantization='bf16',
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

segmentation_results = segmenter.segment_batch(
    images=all_images,
    output_dir='../data/segmentation',
)

print('Segmentation results:')
for i, r in enumerate(segmentation_results):
    print(f'  image {i}: {len(r["objects"])} objects')

if reconstruction_results.get('camera_data'):
    objects_3d = project_objects_to_3d(
        segmentation_results=segmentation_results,
        camera_data=reconstruction_results['camera_data'],
        depth_maps=reconstruction_results.get('depth_maps'),
    )
    print(f'\nProjected {len(objects_3d)} objects to 3-D')
    for obj in objects_3d[:5]:
        print(f'  {obj["label"]} ({obj["category"]}): {[round(v,2) for v in obj["position_3d"]]}')
else:
    objects_3d = []
    print('No camera data — skipping 3-D projection')

## 5. Audio Generation (MMAudio)

In [ ]:
import yaml
from pathlib import Path
from src.audio_generation import AudioGenerator

_cfg_audio_path = Path('../config.yaml')
_da = {}
if _cfg_audio_path.exists():
    with open(_cfg_audio_path) as _f:
        _da = yaml.safe_load(_f).get('audio', {})
audio_gen = AudioGenerator(
    sample_rate=_da.get('sample_rate', 44100),
    use_local_inference=_da.get('use_local_inference', False),
    mmaudio_repo=_da.get('mmaudio_repo'),
)

# Take top 5 objects by confidence
to_process = sorted(objects_3d, key=lambda x: x.get('confidence', 0), reverse=True)[:5]
print(f'Generating audio for {len(to_process)} objects...')

objects_with_audio = audio_gen.generate_batch(
    objects=to_process,
    output_dir='../data/audio',
    duration=3.0,
)

for obj in objects_with_audio:
    status = '✓' if obj.get('audio_file') else '✗'
    print(f'  {status} {obj["label"]}')

# Ambient scene audio
if scene_description:
    ambient = audio_gen.generate_ambient_audio(
        scene_description=scene_description,
        duration=5.0,
        output_path='../data/audio/ambient.wav',
    )
    print(f'Ambient audio: {ambient}')

## 6. Spatial Audio Positioning

In [ ]:
from src.spatial_audio import SpatialAudioPositioner, export_for_unity

positioner = SpatialAudioPositioner(
    coordinate_system='right_handed',
    intensity_falloff='inverse_square',
)

manifest = positioner.create_audio_manifest(
    objects_3d=objects_with_audio,
    camera_data=reconstruction_results.get('camera_data'),
    output_path='../data/output/audio_manifest.json',
)

print(f'Audio manifest: {len(manifest["audio_sources"])} sources, '
      f'{len(manifest["listener_positions"])} listeners')

vis_path = positioner.visualize_audio_scene(
    manifest=manifest,
    point_cloud_path=reconstruction_results.get('dense_point_cloud'),
    output_path='../data/output/audio_visualization.ply',
)
print(f'Visualization: {vis_path}')
print(f'HTML viewer:   {vis_path.replace(".ply", ".html")}')

unity_path = export_for_unity(
    manifest=manifest,
    output_path='../data/output/unity_scene.json',
)
print(f'Unity export:  {unity_path}')

## 7. Visualise 3-D Scene

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

for source in manifest['audio_sources']:
    p = source['position_3d']
    i = source['intensity']
    ax.scatter(p[0], p[1], p[2], c=[[i, 0.2, 1-i]], s=200, marker='o', label=source['label'][:20])

for listener in manifest['listener_positions']:
    p = listener['position']
    ax.scatter(p[0], p[1], p[2], c='green', s=100, marker='^')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Spatial Audio Scene (spheres=sources, triangles=cameras)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

print('\n' + '='*60)
print('PIPELINE COMPLETE')
print('='*60)
print(f'Input images:      {len(images)}')
print(f'Generated images:  {len(generated_images)}')
print(f'Total images:      {len(all_images)}')
print(f'Objects detected:  {sum(len(r["objects"]) for r in segmentation_results)}')
print(f'Objects in 3-D:    {len(objects_3d)}')
print(f'Audio files:       {len([o for o in objects_with_audio if o.get("audio_file")])}')
print(f'Audio sources:     {len(manifest["audio_sources"])}')